# Signal Peptide Efficiency Prediction — Full Reproducibility Notebook

This notebook runs all 10 analysis scripts in sequence, reproducing every result in the manuscript.

**Best result:** 5-seed ensemble MSE of **0.932** [0.823, 1.054] (95% bootstrap CI), surpassing both the physicochemical baseline of 1.22 and a prior benchmark of 0.953.

### Prerequisites

1. Install dependencies: `pip install -r requirements.txt`
2. Place all data files in the `data/` directory (see README for details)
3. Ensure you have sufficient disk space for results and figures

### Runtime Estimates

| Script | Estimated Runtime |
|--------|------------------|
| 01 Baseline reproduction | ~1 min |
| 02 RF hyperparameter search | ~2–8 hrs |
| 03 NN regression search | ~1–2 hrs |
| 04 Final comparison chart | ~10 sec |
| 05 Cross-dataset generalization | ~30 min |
| 06 Vector regression | ~1–2 hrs |
| 07 Design task evaluation | ~5 min |
| 08 Bootstrap CIs | ~30–40 min |
| 09 Vector architecture search | ~2–3 hrs |
| 10 Vector ensemble optimization | ~5 hrs |

Total: approximately 12–22 hours on CPU.

## Script 01: Baseline Reproduction

Reproduces the Grasso et al. random forest baseline using the exact published hyperparameters (75 trees, max depth 25). Expected test MSE: ~1.193 (within 2.2% of the reported 1.22).

In [ ]:
%%time
!python3 -u scripts/01_grasso_reproduction.py

## Script 02: RF Hyperparameter Search

Performs randomized hyperparameter search (100 iterations, 5-fold CV) independently for each of the four feature types: PhysChem, ESM2-650M, ESM2-3B, and Ginkgo-AA0.

**Warning:** This is one of the longest-running scripts (~2–8 hours).

In [ ]:
%%time
!python3 -u scripts/02_rf_hyperparameter_search.py

## Script 03: NN Regression Search

Searches 40 neural network configurations per feature type with dimension-aware architecture grids. Uses 80/20 train/validation split with early stopping. Best result: Ginkgo-AA0 NN with MSE 1.050.

In [ ]:
%%time
!python3 -u scripts/03_nn_regression.py

## Script 04: Final Comparison Chart

Generates a grouped bar chart comparing all 9 scalar model–feature combinations (Table 1 / Figure 2 in the paper). Quick script that reads saved results from Scripts 01–03.

In [ ]:
%%time
!python3 -u scripts/04_final_comparison.py

## Script 05: Cross-Dataset Generalization

Evaluates the best ESM2-650M RF and 5-seed NN ensemble on four external signal peptide datasets (Wu, Xue, Zhang-P43, Zhang-PglVM). Tests whether models trained on Grasso data generalize across organisms and experimental systems.

In [ ]:
%%time
!python3 -u scripts/05_cross_dataset_generalization.py

## Script 06: Vector Regression

Predicts 10-dimensional bin probability distributions (softmax output) instead of scalar WA. Trains 5-seed ensembles with both cross-entropy and focal loss for all three PLM embedding types. Best result: Ginkgo-AA0 + focal loss, MSE 1.001.

In [ ]:
%%time
!python3 -u scripts/06_vector_regression.py

## Script 07: Design Task Evaluation

Tests whether models can predict which designed mutations improve or worsen signal peptide secretion efficiency. Evaluates RF and NN models (physicochemical features) on 4,836 designed variants across 134 genes.

In [ ]:
%%time
!python3 -u scripts/07_design_task_evaluation.py

## Script 08: Bootstrap Confidence Intervals

Computes 95% bootstrap confidence intervals (10,000 resamples) for all 16 models. Produces a forest plot showing MSE point estimates with CI whiskers.

In [ ]:
%%time
!python3 -u scripts/08_bootstrap_ci.py

## Script 09: Vector Architecture Search

Full-data vector regression with no validation split. Explores 3-layer and 4-layer architectures, cosine annealing, and increased ensemble sizes. Identifies (256, 256, 128) as the best architecture with 5-seed ensemble MSE 0.973.

In [ ]:
%%time
!python3 -u scripts/09_vector_architecture_search.py

## Script 10: Vector Ensemble Optimization

Final optimization pass on the best architecture. Tunes dropout rate (0.15–0.35), tests 20-seed ensembles, and mixed-architecture ensembles. Achieves the best overall result: MSE **0.932** with dropout 0.35 and 5-seed ensemble.

**Warning:** This is the longest-running script (~5 hours on CPU).

In [ ]:
%%time
!python3 -u scripts/10_vector_ensemble_optimization.py

## Summary

All 10 scripts have completed. Key outputs:

- **Results:** JSON and CSV files in `results/`
- **Figures:** PNG plots (300 DPI) in `figures/`
- **Paper:** Compile with `cd paper && bash compile.sh`

### Headline Results

| Milestone | MSE | Script |
|-----------|-----|--------|
| Grasso baseline reproduction | 1.193 | 01 |
| Best RF (Ginkgo-AA0) | 1.158 | 02 |
| Best scalar NN (Ginkgo-AA0) | 1.050 | 03 |
| Best vector NN (Ginkgo-AA0 + focal) | 1.001 | 06 |
| Architecture search best | 0.973 | 09 |
| **Final optimized (dropout 0.35)** | **0.932** | **10** |

The final model achieves a 23.6% improvement over the Grasso et al. baseline (1.22) and a 2.2% improvement over the prior benchmark (0.953).